In [ ]:
# ============================================================
# GAN ON MNIST DATASET USING TENSORFLOW / KERAS
# ============================================================
import tensorflow as tf
from tensorflow.keras import layers, Sequential
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. LOAD MNIST DATASET
# ============================================================

(x_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

print("Original MNIST shape:", x_train.shape)
print("Number of training images:", len(x_train))


# ============================================================
# 2. NORMALIZE PIXEL VALUES
#    Convert [0, 255] → [-1, +1]
# ============================================================

x_train = x_train.astype("float32")

x_train = (x_train - 127.5) / 127.5

print("\nMinimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())
# ============================================================
# 3. RESHAPE DATA
#    (60000, 28, 28) → (60000, 28, 28, 1)
# ============================================================
x_train = np.expand_dims(x_train, axis=-1)

print("\nNew shape:", x_train.shape)

# ============================================================
# 4. CREATE DATASET
# ============================================================

BATCH_SIZE = 128

dataset = tf.data.Dataset.from_tensor_slices(x_train)

dataset = dataset.shuffle(60000)

dataset = dataset.batch(BATCH_SIZE)


# ============================================================
# 5. BUILD THE GENERATOR
# ============================================================

LATENT_DIM = 100


def build_generator():

    model = Sequential()

    # --------------------------------------------------------
    # Input: Random noise vector of size 100
    # --------------------------------------------------------
    model.add(layers.Input(shape=(LATENT_DIM,)))

    # Hidden Layer 1
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(negative_slope=0.2))

    # Hidden Layer 2
    model.add(layers.Dense(512))
    model.add(layers.LeakyReLU(negative_slope=0.2))

    # Hidden Layer 3
    model.add(layers.Dense(1024))
    model.add(layers.LeakyReLU(negative_slope=0.2))

    # Output Layer
    # 28 × 28 = 784 pixels
    model.add(layers.Dense(784, activation="tanh"))

    # Convert 784 values into a 28 × 28 image
    model.add(layers.Reshape((28, 28, 1)))

    return model

# Create Generator
generator = build_generator()

print("\n================ GENERATOR ================")
generator.summary()

# ============================================================
# 6. BUILD THE DISCRIMINATOR
# ============================================================

def build_discriminator():

    model = Sequential()

    # Input MNIST image
    model.add(layers.Input(shape=(28, 28, 1)))

    # Convert 28×28 image → 784 values
    model.add(layers.Flatten())

    # Hidden Layer 1
    model.add(layers.Dense(512))
    model.add(layers.LeakyReLU(negative_slope=0.2))

    # Hidden Layer 2
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(negative_slope=0.2))

    # Output Layer
    # Probability that image is REAL
    model.add(layers.Dense(1, activation="sigmoid"))

    return model


# Create Discriminator
discriminator = build_discriminator()

print("\n============== DISCRIMINATOR ==============")
discriminator.summary()


# ============================================================
# 7. DEFINE LOSS FUNCTION
# ============================================================

cross_entropy = tf.keras.losses.BinaryCrossentropy()


# ============================================================
# 8. DEFINE OPTIMIZERS
# ============================================================

generator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)

discriminator_optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0002,
    beta_1=0.5
)


# ============================================================
# 9. DEFINE ONE TRAINING STEP
# ============================================================

@tf.function
def train_step(real_images):

    # Number of real images in current batch
    batch_size = tf.shape(real_images)[0]

    # --------------------------------------------------------
    # STEP 1: CREATE RANDOM NOISE
    # --------------------------------------------------------

    noise = tf.random.normal(
        shape=(batch_size, LATENT_DIM)
    )


    # --------------------------------------------------------
    # STEP 2: TRAIN GENERATOR AND DISCRIMINATOR
    # --------------------------------------------------------

    with tf.GradientTape() as gen_tape, \
         tf.GradientTape() as disc_tape:


        # ----------------------------------------------------
        # GENERATOR CREATES FAKE IMAGES
        # ----------------------------------------------------

        fake_images = generator(
            noise,
            training=True
        )

        # ----------------------------------------------------
        # DISCRIMINATOR PREDICTS REAL IMAGES
        # ----------------------------------------------------

        real_predictions = discriminator(
            real_images,
            training=True
        )


        # ----------------------------------------------------
        # DISCRIMINATOR PREDICTS FAKE IMAGES
        # ----------------------------------------------------

        fake_predictions = discriminator(
            fake_images,
            training=True
        )


        # ====================================================
        # DISCRIMINATOR LOSS
        # ====================================================

        # Real images should be classified as 1
        real_loss = cross_entropy(
            tf.ones_like(real_predictions),
            real_predictions
        )


        # Fake images should be classified as 0
        fake_loss = cross_entropy(
            tf.zeros_like(fake_predictions),
            fake_predictions
        )


        # Total discriminator loss
        discriminator_loss = real_loss + fake_loss


        # ====================================================
        # GENERATOR LOSS
        # ====================================================

        # Generator wants discriminator to say:
        #
        # Fake image = REAL = 1
        #
        generator_loss = cross_entropy(
            tf.ones_like(fake_predictions),
            fake_predictions
        )


    # ========================================================
    # 10. CALCULATE GRADIENTS
    # ========================================================

    generator_gradients = gen_tape.gradient(
        generator_loss,
        generator.trainable_variables
    )

    discriminator_gradients = disc_tape.gradient(
        discriminator_loss,
        discriminator.trainable_variables
    )


    # ========================================================
    # 11. UPDATE GENERATOR WEIGHTS
    # ========================================================

    generator_optimizer.apply_gradients(
        zip(
            generator_gradients,
            generator.trainable_variables
        )
    )


    # ========================================================
    # 12. UPDATE DISCRIMINATOR WEIGHTS
    # ========================================================

    discriminator_optimizer.apply_gradients(
        zip(
            discriminator_gradients,
            discriminator.trainable_variables
        )
    )


    return generator_loss, discriminator_loss


# ============================================================
# 13. FUNCTION TO DISPLAY GENERATED IMAGES
# ============================================================

def generate_and_show_images(epoch, num_images=16):

    # Create random noise
    noise = tf.random.normal(
        shape=(num_images, LATENT_DIM)
    )

    # Generator creates new images
    generated_images = generator(
        noise,
        training=False
    )

    # Convert TensorFlow tensor to NumPy
    generated_images = generated_images.numpy()

    # Convert [-1, 1] → [0, 1]
    generated_images = (
        generated_images + 1
    ) / 2


    # Display images
    plt.figure(figsize=(8, 8))

    for i in range(num_images):

        plt.subplot(4, 4, i + 1)

        plt.imshow(
            generated_images[i, :, :, 0],
            cmap="gray"
        )

        plt.axis("off")

    plt.suptitle(
        f"Generated MNIST Digits - Epoch {epoch}"
    )

    plt.show()


# ============================================================
# 14. TRAIN THE GAN
# ============================================================

EPOCHS = 50


for epoch in range(EPOCHS):

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}"
    )

    for real_images in dataset:

        gen_loss, disc_loss = train_step(
            real_images
        )


    # Print losses
    print(
        "Generator Loss:",
        float(gen_loss)
    )

    print(
        "Discriminator Loss:",
        float(disc_loss)
    )


    # Display generated images every 5 epochs
    if (epoch + 1) % 5 == 0:

        generate_and_show_images(
            epoch + 1
        )


# ============================================================
# 15. GENERATE FINAL NEW DIGITS
# ============================================================

print("\nGenerating final MNIST images...")

generate_and_show_images(
    EPOCHS,
    num_images=16
)


Original MNIST shape: (60000, 28, 28)
Number of training images: 60000

Minimum pixel value: -1.0
Maximum pixel value: 1.0

New shape: (60000, 28, 28, 1)

================ GENERATOR ================


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 256)                 │          25,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu (LeakyReLU)              │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 512)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_1 (LeakyReLU)            │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1024)                │         525,312 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_2 (LeakyReLU)            │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 784)                 │         803,600 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ reshape (Reshape)                    │ (None, 28, 28, 1)           │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,486,352 (5.67 MB)

 Trainable params: 1,486,352 (5.67 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 512)                 │         401,920 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_3 (LeakyReLU)            │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 256)                 │         131,328 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ leaky_re_lu_4 (LeakyReLU)            │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 1)                   │             257 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 533,505 (2.04 MB)

 Trainable params: 533,505 (2.04 MB)

 Non-trainable params: 0 (0.00 B)


Epoch 1/50
Generator Loss: 0.8991751670837402
Discriminator Loss: 1.3306385278701782

Epoch 2/50
Generator Loss: 0.6744487881660461
Discriminator Loss: 0.9072274565696716

Epoch 3/50
Generator Loss: 0.9495241045951843
Discriminator Loss: 0.8557076454162598

Epoch 4/50
